<a href="https://colab.research.google.com/github/HarshtJhalani/COMPANY-POLICY-AI-ASSISTANT/blob/main/COMPANY%20POLICY%20AI%20ASSISTANT" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU \
    langchain_classic \
    langchain_google_genai \
    langchain_community \
    langchain_huggingface \
    langchain_chroma \
    langchain_experimental \
    sentence-transformers \
    chromadb \
    pypdf

In [ ]:
from langchain_chroma import Chroma

In [ ]:
import os
import re
import warnings
import pandas as pd
import numpy as np

from google.colab import drive, userdata

warnings.filterwarnings("ignore")

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
pdf_directory = "/content/drive/MyDrive/Datasets/Company_documents"

In [ ]:
from google.colab import userdata

# Get Gemini API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("keyy")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.2
)

In [ ]:
pdf_directory = "/content/drive/MyDrive/Datasets/Company_documents"

pdf_paths = [
    os.path.join(pdf_directory, f)
    for f in os.listdir(pdf_directory)
    if f.lower().endswith(".pdf")
]

print("PDF files found:")
for path in pdf_paths:
    print("-", os.path.basename(path))

print(f"\nTotal documents: {len(pdf_paths)}")

PDF files found:
- Data_Axle_HR_Policies.pdf
- Data_Axle_Company_Policies.pdf
- Data_Axle_2026_Indian_Holidays.pdf
- Data_Axle_Leave_Policies.pdf

Total documents: 4


In [ ]:
pdf_docs = []

for path in pdf_paths:

    print(f"Loading: {os.path.basename(path)}")

    loader = PyPDFLoader(path)

    docs = loader.load()

    # Add document-level metadata
    for doc in docs:
        doc.metadata["document_name"] = os.path.basename(path)

    pdf_docs.extend(docs)

print("\nTotal pages loaded:", len(pdf_docs))

Loading: Data_Axle_HR_Policies.pdf
Loading: Data_Axle_Company_Policies.pdf
Loading: Data_Axle_2026_Indian_Holidays.pdf
Loading: Data_Axle_Leave_Policies.pdf

Total pages loaded: 15


In [ ]:
#Previewing
print("Document:", pdf_docs[0].metadata.get("document_name"))
print("Page:", pdf_docs[0].metadata.get("page"))

print("\nText preview:")
print(pdf_docs[0].page_content[:250])

Document: Data_Axle_HR_Policies.pdf
Page: 0

Text preview:
HR Policies
Employment Terms, Compensation, Benefits, Performance and Separation Guidelines for Data Axle
India
Document ID: DA-IN-HR-002
Version: 4.0
Effective Date: 1 January 2026
Policy Owner: Head of People Operations, India (Priya Deshmukh)
Appr


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(pdf_docs)

print("Total chunks:", len(chunks))

Total chunks: 114


In [ ]:
for i, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = f"chunk_{i}" #assigning unique id to each chunk

    chunk.metadata["source_file"] = os.path.basename(    #stored pdf file name with each chunk
        chunk.metadata.get("source", "Unknown")
    )

In [ ]:
#creating embeddings and storing in chroma DB
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="Company_documents"
)

print("Chroma vector database created successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma vector database created successfully.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [ ]:
def retrieve_policy_context(question):

    documents = retriever.invoke(question)

    context_parts = []

    for i, doc in enumerate(documents, 1):

        source = doc.metadata.get(
            "document_name",
            "Unknown Document"
        )

        page = doc.metadata.get(
            "page",
            0
        )

        context_parts.append(
            f"""
SOURCE {i}
Document: {source}
Page: {page + 1}

Content:
{doc.page_content}
"""
        )

    return "\n".join(context_parts), documents

In [ ]:
#hallucination prevention

RAG_PROMPT = """
You are the Company Policy Answering Engine.

Answer the employee's question ONLY using the policy
information provided below.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent company policies.
3. Do not assume information that is not explicitly present.
4. If the provided information does not answer the question,
   respond exactly that the information could not be found
   in the provided company policy documents.
5. If multiple policies are relevant, combine them carefully.
6. Mention important conditions, eligibility requirements,
   exceptions, or limitations.
7. Cite the document and page number used.
8. Keep the answer clear and concise.

POLICY CONTEXT:

{context}

EMPLOYEE QUESTION:

{question}

ANSWER:
"""

In [ ]:
def generate_policy_answer(question):

    context, documents = retrieve_policy_context(question)

    prompt = RAG_PROMPT.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    answer = response.content

    return {
        "answer": answer,
        "documents": documents
    }

In [ ]:
result = generate_policy_answer(
    "How many days of leave can an employee take?"
)

print(result["answer"])

[{'type': 'text', 'text': 'Based on the provided policy documents, the exact total number of leave days an employee can take across all leave types is not specified, as entitlements for Casual Leave (CL) and Sick Leave (SL) are not detailed. However, the available documents outline the following specific entitlements and limitations:\n\n* **Earned Leave (EL):** \n  * **Entitlement:** 18 days in a full calendar year.\n  * **Accrual & Conditions:** Accrues at the rate of 1.5 days per completed calendar month of service and is credited on the first day of the following month. EL is granted in units of a full day only.\n  * *Citation:* `Data_Axle_Leave_Policies.pdf`, Page 1\n\n* **Compensatory Day Off:** \n  * **Entitlement:** 1 day off earned for working at least 4 hours on a Fixed Holiday or selected Restricted Holiday.\n  * **Conditions:** Must be recorded in Axle Connect within 7 days and used within 60 calendar days of the holiday worked. It cannot be carried beyond 60 days, encashed,

In [ ]:
result = generate_policy_answer(
    "What is formula 1?"
)

print(result["answer"])

[{'type': 'text', 'text': 'The information could not be found in the provided company policy documents.', 'extras': {'signature': 'Et4WCtsWAWkUfRP7TbfPk4r+VR7U7NVR4Vx+G8MSWxzmOsMn8hVmicAbMiLPYZzIxd1k5qg3QoquvdWfXUmnDp11yiAtp6L2zyxPGvDySqnVLPZhQPocO7w2s0cTXVwpSydxp0Nxg9NoPUTXt7jZA4TRKjGaCFb9UHkoC/qIwNW1vQp8Yg76FE9RdiHaQFfA6j9UcRlH034/aoD1XprwyfR8xyiBEfCO3ieMJCae3cazsEE7stXbjH4Bsysxd0e/cKiq52mdLeQfLIabLwEu8pjFwsmbDfbN6ypMeASqq/RTiXLAOuigAhTGKHgoK3jhh6T8Zu2k00RBBqN+cj6OoJzYyVUNwrjmLC/tpBdCHXDmOYv+u4VBS6wVoWkD4vxsnj1369Q+v4P/yXawsaLbS5i/fi9dmdyq7yEjxypQ6jgcjsZfNda4nZt/YLR2VVx3mJjawTCCsW0KmH68RlJIaH2VP8WUCVHlQKhYwGEIF1KDYo0nN060Wu6HsncJ3zaZvQ6vWEwNOOsORSQaD50nESH7mbne308PwltURP2pAbz4G4FKz1JwJ5pQpFvwjRRvMiTIYo++Lmt2vQZJEiOJ90LKbHjsqKHfYTFKqgHbpDN+5vNLVNZFpdZ2t/cKFLXwyOk7HnmJ8L/ncIPpQV5PKTxdlKOMZPwsb+60aw7a8BgIITsOH+gIC3iu7jIoYZsUIqWOFD0etY0PzpqbedVIj67z9JxkBdIUzw6sB7d29qBB9z9Zm9iTru84GbY3IZvDMwFnV3WLfmToZK9R7qL8uvY5r96NLtpHAX5Zu9ZB8Vt219ww2HrFJxMNh9EykYsNXtdDi4Iv0HWUFo45Hwrdj1l56j+1kwm4w3w3P

In [ ]:
@tool
def company_policy_rag(question: str) -> str:
    """
    Search company policy documents and answer an employee's
    question using only the retrieved policy information.
    Use this tool when the employee needs a factual answer
    from company policy documents.
    """

    result = generate_policy_answer(question)

    return result["answer"]

In [ ]:
@tool
def search_policy_documents(query: str) -> str:
    """
    Search the company policy documents and return relevant
    document excerpts with their document names and page numbers.
    Use this when evidence or source information needs to be inspected.
    """

    documents = retriever.invoke(query)

    if not documents:
        return "No relevant policy documents were found."

    results = []

    for i, doc in enumerate(documents, 1):

        source = doc.metadata.get(
            "document_name",
            "Unknown Document"
        )

        page = doc.metadata.get(
            "page",
            0
        )

        results.append(
            f"""
Result {i}
Document: {source}
Page: {page + 1}

Excerpt:
{doc.page_content[:800]}
"""
        )

    return "\n".join(results)

In [ ]:
@tool
def calculate_policy_value(expression: str) -> str:
    """
    Perform basic arithmetic calculations needed when answering
    policy-related numerical questions.

    Example:
    '12 - 5'
    '20 * 0.5'
    """

    allowed = re.fullmatch(
        r"[0-9+\-*/().\s]+",
        expression
    )

    if not allowed:
        return "Invalid mathematical expression."

    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return f"Calculation result: {result}"

    except Exception:
        return "Unable to perform calculation."

In [ ]:
@tool
def get_remaining_holidays(
    start_date: str = None,
    end_date: str = "2026-12-31"
) -> str:
    """
    Retrieve the company holiday policy/document and identify
    holidays occurring between the specified dates.

    The start_date and end_date can be in any readable format.
    If start_date is not specified or is empty, today's date will be used.
    """
    from datetime import datetime
    import pandas as pd

    # Handle start_date defaulting to today if not provided
    if not start_date or str(start_date).strip() == "":
        start_dt = datetime.now()
        start_date_str = start_dt.strftime("%Y-%m-%d")
    else:
        try:
            # Use pandas to flexibly parse input dates in any format
            start_dt = pd.to_datetime(start_date)
            start_date_str = start_dt.strftime("%Y-%m-%d")
        except Exception:
            start_date_str = datetime.now().strftime("%Y-%m-%d")

    try:
        end_dt = pd.to_datetime(end_date)
        end_date_str = end_dt.strftime("%Y-%m-%d")
    except Exception:
        end_date_str = "2026-12-31"

    query = (
        "Data Axle 2026 Indian holidays public holidays "
        "national holidays holiday dates"
    )

    documents = retriever.invoke(query)

    if not documents:
        return "No holiday information was found in the company documents."

    holiday_context = []

    for doc in documents:
        source = doc.metadata.get(
            "document_name",
            "Unknown Document"
        )
        page = doc.metadata.get("page", 0)
        holiday_context.append(
            f"""
Document: {source}
Page: {page + 1}

{doc.page_content}
"""
        )

    context = "\n".join(holiday_context)

    prompt = f"""
You are extracting holiday information from company documents.

Use ONLY the information provided below.

Find all company/public/national holidays that fall
between {start_date_str} and {end_date_str}, inclusive.

Return:
1. Holiday name
2. Date
3. Day of week
4. Source document
5. Page number

Do not invent or add holidays that are not present
in the provided documents.

If the documents do not contain enough information,
say so clearly.

DOCUMENTS:

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
agent = create_agent(
    model = llm,
    tools = [company_policy_rag, search_policy_documents, calculate_policy_value,get_remaining_holidays]
)

In [ ]:
AGENT_INSTRUCTIONS = """
You are an AI-powered Company Policy Assistant.

Your job is to help employees understand information
contained in the provided company policy documents.

You have access to four tools:

1. company_policy_rag
   Use this to answer factual questions using the
   company policy documents.

2. search_policy_documents
   Use this to inspect policy evidence, source documents,
   page numbers, or relevant excerpts.

3. calculate_policy_value
   Use this only for arithmetic calculations that are
   based on information retrieved from the policies.

4. get_remaining_holidays
   Use this when the employee asks about:
   - remaining national or public holidays
   - holidays between two dates
   - holidays remaining until a specific date
   - the number of remaining holidays
   - upcoming holidays listed in the company holiday calendar

   This tool must obtain holiday dates from the
   Data_Axle_2026_Indian_Holidays.pdf document.
   Never use hard-coded holiday dates or outside knowledge.

IMPORTANT RULES:

- Company policy documents are the ONLY source of truth.
- Never use outside knowledge for company policy questions.
- Never invent a policy.
- Never assume an eligibility condition.
- When information cannot be found, clearly say so.

- For holiday-related questions, use the
  get_remaining_holidays tool instead of relying on
  general knowledge about Indian or public holidays.

- Holiday dates must be retrieved from the company's
  holiday document.

- The current date should be determined dynamically.
  Never assume that "today" is a fixed date.

- For questions asking how many holidays remain,
  first retrieve the holiday dates from the document,
  then calculate which dates fall within the requested range.

- For numerical questions involving policy information,
  retrieve the relevant policy information before
  performing calculations.

- Use calculate_policy_value only after the relevant
  policy information has been retrieved.

- When multiple policies are relevant, consider all of them.

- Give concise answers.

- Mention the relevant policy document and page whenever
  possible.

- If the requested information is not present in the
  company documents, clearly state that it could not
  be found in the provided company policy documents.

- Do not fabricate dates, numbers, eligibility conditions,
  exceptions, or policy rules.
"""


In [ ]:
def ask_company_policy_agent(question):

    response = agent.invoke({
        "messages": [
            {
                "role": "system",
                "content": AGENT_INSTRUCTIONS
            },
            {
                "role": "user",
                "content": question
            }
        ]
    })

    return response

In [ ]:
def ask_policy(question):

    response = ask_company_policy_agent(question)

    messages = response["messages"]

    final_message = messages[-1]

    print("\n" + "="*80)
    print("EMPLOYEE QUESTION")
    print("="*80)
    print(question)

    print("\n" + "="*80)
    print("POLICY ASSISTANT")
    print("="*80)
    print(final_message.content)

    return response

In [ ]:
print(ask_policy("How many leaves in a year, please give categorized, sick and normal/casual")['messages'][-1].content[0]['text'])


EMPLOYEE QUESTION
How many leaves in a year, please give categorized, sick and normal/casual

POLICY ASSISTANT
[{'type': 'text', 'text': 'According to the **Data Axle Leave Policies** document (*Data_Axle_Leave_Policies.pdf*, Pages 1–2), eligible employees are entitled to a total of **36 days of paid leave** per calendar year, categorized as follows:\n\n* **Earned Leave (EL):** **18 days** per full calendar year (accrues at 1.5 days per completed month) (*Page 1*)\n* **Casual Leave (CL):** **8 days** per full calendar year (credited in advance on 1 January) (*Page 2*)\n* **Sick Leave (SL):** **10 days** per full calendar year (credited in advance on 1 January) (*Page 2*)', 'extras': {'signature': 'EvkICvYIAWkUfRNrPZ+WTWsBJWgkTgV2lVo+LhaQ/6akGFOFN1MT5A83iOMxhJ6kLU5q9XC+4+lIhbL6WxOqmnv8vDjaAkopOFxq+4GM1nkjcAIapgPcpVMQK4ZFCZFapzgV19lb4NjiRtT7f6SOPbVYMA6urulV1nElpbkpVijEOrU8tX2n5t8jDrDy6ik3bE3urUbJONdxTjPz8TQ9V2KSoy3yeUoZeLu0QC+5uqxxPvChNPWtJPK4H9dJfBcH2/RQDIiRIzvdpRjYta765dcDAyHro2CKHNbt

In [ ]:
print(ask_policy("Where is WFH policy mentioned")['messages'][-1].content[0]['text'])


EMPLOYEE QUESTION
Where is WFH policy mentioned

POLICY ASSISTANT
[{'type': 'text', 'text': 'The Work From Home (WFH) and Hybrid Work policy is primarily detailed in:\n\n* **Data_Axle_Company_Policies.pdf** (Page 3) under **Section 5: Working Hours, Hybrid Work and Work From Home** (specifically **Section 5.2: Hybrid Working Model** and **Section 5.3: Work From Home Conditions**).\n\nIt is also referenced in:\n* **Data_Axle_HR_Policies.pdf** (Page 2) under **Section 6: Working Hours, Shifts and Overtime**.', 'extras': {'signature': 'Eu8FCuwFAWkUfRP+anw9BDJO8EJVU/wxzSxiix94rl6IzQBb/XdkhafXqXy0I53zU6x1ka1kjuqSvGJbwNGBOjxwLa4R6U4wVCv8CPpJi4PADVOEn4NGxN0PiMF/bNf+52Ji7wBA9TaPqcu/n21dOMgG1H+WEqgAP8+TPBGj2TFdpC71pNKQZscOTrBgBgDjY1kasIcq44iZwdN026rsuxBchszMOwCcmUgZcdl9/7TjfR4vc7DIDoyukDVO5IA+npT+kvOsHSNChzIWjejQhfPp76CZrqd4CNw4svEdquULaPbqQKpuH9b+aI5Lp8DA8KT0s2AkiLUlbWTxpFjnckOLeSXICI860iSvhmYCDyAazyt+ho8/T80XZ8cuF8MU7cwK9/KAQrOPRZ2uZWkgt8TMFU+sxAXqQtvy3Qz1iHpHcj4Xvo5cpiBlmnXco/nhr+a6W1XqKi6+

In [ ]:
print(ask_policy("If an employee receives 12 annual leave days and has already "
    "used 5 days, how many days are remaining?")['messages'][-1].content[0]['text'])


EMPLOYEE QUESTION
If an employee receives 12 annual leave days and has already used 5 days, how many days are remaining?

POLICY ASSISTANT
[{'type': 'text', 'text': 'An employee with 12 annual leave days who has used 5 days has **7 days** remaining (12 - 5 = 7).', 'extras': {'signature': 'EpQGCpEGAWkUfRPeugLuri084OcvBir2Wa5j/v+B9yIDRbVEuPBZ3u2Z9Xs9AKyNjFw43ToP2R3Whg6j8Mmux7LNlZpqayHDWIHrrm/qYYxEv/9y965HadfL+BbPPDbtD7y/MHrpktX6xcuoeiGmwp2ekWbYl1RLrSs6Pz/PPoJ+vETR+qL1qlGWdhF8p/tM2C/FwBjewZ4ELdc0wIihdWNTDdxl4rgApM3Klahehk7s9Uj2tfhuFxBYpRHHJ5yM9Jl8OCO+Qi3IE6oyHmE0DZFuO7gTsdCoAXXWBHAxucDm5Mdic9THs5czOZZ/ZMLrVzLCDqTTXcvk5w1ejB9fs+tiwP+aWXQ16UqTXI/zrdT9SHgfwE22JMEkkPKe7He5M3VlqKcmj2XI6vKkMMrIFs+SoDXB7m9ziK7G9uZkU9vjVwsZbUfcS18DuNkH0/tcz4evqH0mYrHnBaAIvBeT1JYd5A8Lg/Z7b8JSqNW3FhuISc2VxbXewOMi24A0pzpiF2ZBUaK29jQTemigBukPUEl9JUQqjBgOaLg40GMKT+6Wa3MbVSbeortCt85YfdpMcvVioAhxRh05xZGb0AzesFT2dTK7hjNOlfwIcVdy6PDDP5RIdTTwUw5Cn4nPfoQo06hvcSC+dguV34lI9oEK5FuXnuwh1eRe25YFx5AoEpvuip1I+/Aw4U3tmR7KBaocz6NGD

In [ ]:
ask_policy("Does the company provide free gym memberships?")['messages'][-1].content[0]['text']


EMPLOYEE QUESTION
Does the company provide free gym memberships?

POLICY ASSISTANT
[{'type': 'text', 'text': 'Based on the provided company policy documents, there is no mention of free gym memberships or gym reimbursements. This information could not be found in the provided company policy documents.', 'extras': {'signature': 'Eq0MCqoMAWkUfRN9wM53igyOJvuJ2ZOrjAstIwAaDkSVuIdSUiNKbrqVaCwY76BG3mHYdBzTmI2OxVg73e8wvTFodepWSF4pjTmECut3jBRtk4Fr2edyr1lX9sGCKYQum52ABEskrRQKRztzO8GLEHIkINwLoVzsA3qsCh5jkBUKhPWoMP+oIvDNG5AFgKKKJM5EPZZKxC8GRV3wcI7JR699Q9jfhKLgiPVfyFkgIzCWLrju6CQ6d3obiYyu5GYfZCmPvwCjzIqQCumMsFq3QlAn9TbZ3nuhyG/HjLSCGoEZj9IPrYQ+s895LnnxDi+vZTX8DqLiInOON65pyYaI6mdyFJMlYZ46FdX5oQ06/aePuYZpV8cFUutWG2akSeJpvI3bqHx82j1I8ntlkgfhC7aYSkQoR5XogUezEu984AHRgRAHNvnN/dLa5gQkWbmr6LTEla9BLqdReTQTnX/5M5COOBtmvYhEo//DuEA0AcIz7Yv4Ml+IncFDGJG/SZn7YNLbGBHbmfwXZqxRPHHxGGE5T/472qC8GY21dU3Cva3yWqNYWj+rQGR9LSlwPBvuUteWHkLo0q5n/dVpfqOGOlnk3AgTQPSd85SdOmGNoUfBZ37bu2fiNWxrlTY+JMFTJ+we+ni1rLtaoyFukuIeAq/KtG8ea

'Based on the provided company policy documents, there is no mention of free gym memberships or gym reimbursements. This information could not be found in the provided company policy documents.'

In [ ]:
def inspect_agent_response(response):

    print("\n" + "="*80)
    print("AGENT ACTIVITY")
    print("="*80)

    for i, message in enumerate(response["messages"]):

        print(f"\nMessage {i + 1}")
        print("-"*60)

        print("Type:", type(message).__name__)

        if hasattr(message, "tool_calls") and message.tool_calls:
            print("Tool calls:")

            for call in message.tool_calls:
                print(
                    f"  → {call['name']}"
                )
                print(
                    f"    Arguments: {call['args']}"
                )

        if hasattr(message, "content") and message.content:
            if isinstance(message.content, str):
                print(
                    "Content:",
                    message.content[:500]
                )

In [ ]:
response = ask_company_policy_agent(
    "How many casual leaves are available?"
)

inspect_agent_response(response)


AGENT ACTIVITY

Message 1
------------------------------------------------------------
Type: SystemMessage
Content: 
You are an AI-powered Company Policy Assistant.

Your job is to help employees understand information
contained in the provided company policy documents.

You have access to four tools:

1. company_policy_rag
   Use this to answer factual questions using the
   company policy documents.

2. search_policy_documents
   Use this to inspect policy evidence, source documents,
   page numbers, or relevant excerpts.

3. calculate_policy_value
   Use this only for arithmetic calculations that are
   ba

Message 2
------------------------------------------------------------
Type: HumanMessage
Content: How many casual leaves are available?

Message 3
------------------------------------------------------------
Type: AIMessage
Tool calls:
  → company_policy_rag
    Arguments: {'question': 'How many casual leaves are available?'}

Message 4
-----------------------------------------

In [ ]:
print(ask_policy("how many leaves of national and public holidays are remaining from today til dec 31 2026")['messages'][-1].content[0]['text'])


EMPLOYEE QUESTION
how many leaves of national and public holidays are remaining from today til dec 31 2026

POLICY ASSISTANT
[{'type': 'text', 'text': "Based on the company's holiday calendar (**Data_Axle_2026_Indian_Holidays.pdf**, Document ID: `DA-IN-HR-004`, Pages 1–2), there are **3 Fixed (Public/National) Holidays** remaining between today (11 October 2026) and 31 December 2026:\n\n### **Remaining Fixed Holidays:**\n1. **Tuesday, 20 October 2026** – Dussehra (Vijayadashami)\n2. **Monday, 9 November 2026** – Diwali (Balipratipada / Diwali Padwa)\n3. **Friday, 25 December 2026** – Christmas Day\n\n---\n\n### **Remaining Restricted (Optional) Holidays:**\nIn addition to the fixed public holidays, employees may select optional restricted holidays (Section 4, Page 2). The remaining restricted holidays in this date range are:\n* **Tuesday, 10 November 2026** – Bhau Beej (Bhai Dooj)\n* **Tuesday, 24 November 2026** – Guru Nanak Jayanti", 'extras': {'signature': 'ErAKCq0KAWkUfRP5VyKYJ8YK